In [1]:
import numpy as np

class LoopGridworld:
    def __init__(self):
        self.num_states = 5
        self.state = 0

    def reset(self):
        self.state = 0
        return self.state

    def step(self, action):
        if action == 0 and self.state > 0:
            self.state -= 1
        elif action == 1 and self.state < self.num_states - 1:
            self.state += 1

        if self.state == 4:
            return self.state, 10.0, True
        else:
            return self.state, -0.5, False


In [2]:
def generate_episode(env, policy):
    episode = []
    state = env.reset()
    done = False

    while not done:
        action = np.random.choice(len(policy[state]), p=policy[state])
        next_state, reward, done = env.step(action)
        episode.append((state, action, reward))
        state = next_state

    return episode

def every_visit_mc_prediction(env, policy, num_episodes, gamma=1.0):
    V = np.zeros(env.num_states)
    returns = {s: [] for s in range(env.num_states)}

    for _ in range(num_episodes):
        episode = generate_episode(env, policy)
        G = 0

        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]
            G = gamma * G + reward

            returns[state].append(G)
            V[state] = np.mean(returns[state])

    return V


In [3]:
if __name__ == "__main__":
    env = LoopGridworld()

    stochastic_policy = {
        0: [0.0, 1.0],
        1: [0.3, 0.7],
        2: [0.3, 0.7],
        3: [0.3, 0.7],
        4: [0.5, 0.5]
    }

    estimated_values = every_visit_mc_prediction(env, stochastic_policy, num_episodes=5000, gamma=0.95)

    print("Estimated State Values V(s) (Every-Visit):")
    for state, val in enumerate(estimated_values):
        print(f"State {state}: {val:.4f}")


Estimated State Values V(s) (Every-Visit):
State 0: 4.6788
State 1: 5.4309
State 2: 6.9101
State 3: 8.8401
State 4: 0.0000
